# Cost & Efficiency Evaluation

How do you pick the right LLM for production? It's not just about quality — cost and speed matter too.

This notebook compares **8B (fast/cheap)** vs **70B (smart/expensive)** across:

| Section | Question it answers |
|---|---|
| **1. Latency** | How fast does each model respond? |
| **2. Token Efficiency** | How verbose is each model? |
| **3. Cost Estimation** | How much does each query cost across providers? |
| **4. Quality per Dollar** | Is the expensive model worth it? |

## Setup

In [3]:
import os, time, re, json
import numpy as np
import tiktoken
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

groq_client = Groq(api_key=os.getenv('GROQ_API_KEY'))

MODELS = [
    ('8B (Fast)',   'llama-3.1-8b-instant'),
    ('70B (Smart)', 'llama-3.3-70b-versatile'),
]

# Pricing per 1M tokens (input / output)
PRICING = {
    'GPT-4o':              {'input': 2.50,  'output': 10.00},
    'GPT-4o-mini':         {'input': 0.15,  'output': 0.60},
    'Claude Sonnet 4':     {'input': 3.00,  'output': 15.00},
    'Claude Haiku 3.5':    {'input': 0.80,  'output': 4.00},
    'Llama 3.1 8B (Groq)': {'input': 0.05,  'output': 0.08},
    'Llama 3.3 70B (Groq)':{'input': 0.59,  'output': 0.79},
}

def groq_call(prompt, model, system='You are a helpful assistant.', max_tokens=400):
    """Returns full response object (for usage stats)."""
    return groq_client.chat.completions.create(
        model=model,
        messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}],
        temperature=0.0, max_tokens=max_tokens
    )

def parse_json(raw):
    raw = re.sub(r'```json|```', '', raw).strip()
    try: return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group()) if m else {}

def estimate_cost(prompt_tok, compl_tok, pricing):
    return (prompt_tok * pricing['input'] + compl_tok * pricing['output']) / 1_000_000

enc = tiktoken.get_encoding('cl100k_base')

print('Ready!')

Ready!


## 1. Latency

How fast does each model respond? We measure:
- **E2E Latency** — wall-clock time from request to full response
- **Tokens/sec** — how fast the model generates output
- **Queue Time** — time waiting before processing starts (Groq reports this)

In [4]:
prompts = [
    ('Short',  'What is 2+2? Answer in one word.'),
    ('Medium', 'Explain how a CPU works in 3 sentences.'),
    ('Long',   'Compare Python vs Rust: syntax, performance, memory safety, and use cases.'),
]

print('Latency Profiling')
print('=' * 80)
print(f'{"Model":<15} {"Prompt":<8} {"E2E (s)":>8} {"Queue (s)":>10} {"Tokens":>8} {"Tok/s":>8}')
print('-' * 80)

for model_name, model_id in MODELS:
    for ptype, ptext in prompts:
        t0 = time.time()
        resp = groq_call(ptext, model_id)
        e2e = time.time() - t0

        usage = resp.usage
        compl_tok = usage.completion_tokens
        queue = getattr(usage, 'queue_time', 0) or 0
        total = getattr(usage, 'total_time', e2e) or e2e
        tps = compl_tok / total if total > 0 else 0

        print(f'{model_name:<15} {ptype:<8} {e2e:8.3f} {queue:10.4f} {compl_tok:8d} {tps:8.1f}')

Latency Profiling
Model           Prompt    E2E (s)  Queue (s)   Tokens    Tok/s
--------------------------------------------------------------------------------
8B (Fast)       Short       0.217     0.0069        3    208.0
8B (Fast)       Medium      0.277     0.0055      101    692.5
8B (Fast)       Long        0.716     0.0080      400    666.4
70B (Smart)     Short       0.103     0.0157        3    167.8
70B (Smart)     Medium      0.410     0.0297      107    347.1
70B (Smart)     Long        1.544     0.2082      400    319.0


## 2. Token Efficiency

Verbose models waste tokens (= money). We compare each model's response length against a concise gold-standard answer.

- **Ratio** — response tokens / gold tokens. Ratio of 2x = twice as verbose as needed
- **Info Density** — unique content words / total words. Higher = less filler

In [5]:
STOP_WORDS = {'the','a','an','is','are','was','were','be','been','have','has','had',
              'do','does','did','will','would','shall','should','may','might','can','could',
              'to','of','in','for','on','with','at','by','from','as','it','its','this',
              'that','and','but','or','not','no','so','if','then','than','very'}

def info_density(text):
    words = re.findall(r'\b[a-z]+\b', text.lower())
    content = [w for w in words if w not in STOP_WORDS]
    return len(set(content)) / len(words) if words else 0

tests = [
    {'q': 'What is photosynthesis? Answer in one sentence.',
     'gold': 'Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose and oxygen.'},
    {'q': 'Name three benefits of exercise. Be concise.',
     'gold': 'Exercise improves cardiovascular health, boosts mood, and aids weight management.'},
    {'q': 'Explain gravity in exactly two sentences.',
     'gold': 'Gravity is the force that attracts objects with mass toward each other. It keeps planets in orbit and holds us on Earth.'},
]

print('Token Efficiency')
print('=' * 75)

for model_name, model_id in MODELS:
    print(f'\n  {model_name}:')
    for t in tests:
        answer = groq_call(t['q'], model_id, max_tokens=200).choices[0].message.content.strip()
        gold_tok = len(enc.encode(t['gold']))
        resp_tok = len(enc.encode(answer))
        ratio = resp_tok / gold_tok
        density = info_density(answer)
        flag = 'VERBOSE' if ratio > 2.0 else 'OK'
        print(f'    [{flag:>7}] {t["q"][:45]:<45} {ratio:.1f}x  density={density:.0%}')

Token Efficiency

  8B (Fast):
    [     OK] What is photosynthesis? Answer in one sentenc 1.7x  density=60%
    [     OK] Name three benefits of exercise. Be concise.  1.7x  density=85%
    [VERBOSE] Explain gravity in exactly two sentences.     2.8x  density=52%

  70B (Smart):
    [     OK] What is photosynthesis? Answer in one sentenc 1.7x  density=60%
    [     OK] Name three benefits of exercise. Be concise.  1.5x  density=82%
    [VERBOSE] Explain gravity in exactly two sentences.     3.0x  density=52%


## 3. Quality per Dollar

The cheapest model isn't always the best deal. We use an LLM judge to score quality, then compute **quality per dollar**.

This answers: *Is the 70B model worth 10x the price of the 8B?*

In [6]:
JUDGE_SYS = '''Score the response on helpfulness, correctness, completeness (each 1-5).
Return JSON only: {"helpfulness": <int>, "correctness": <int>, "completeness": <int>}'''

MODEL_PRICING = {
    '8B (Fast)':   {'input': 0.05,  'output': 0.08},
    '70B (Smart)': {'input': 0.59,  'output': 0.79},
}

eval_prompts = [
    'Explain how a hash table works and when you would use one.',
    'What is the difference between authentication and authorization?',
    'Describe the CAP theorem and its implications for distributed systems.',
    'How does garbage collection work in Java?',
]

print('Quality per Dollar')
print('=' * 65)

results = {}
for model_name, model_id in MODELS:
    scores, costs = [], []
    for prompt in eval_prompts:
        resp = groq_call(prompt, model_id)
        answer = resp.choices[0].message.content.strip()
        cost = estimate_cost(resp.usage.prompt_tokens, resp.usage.completion_tokens, MODEL_PRICING[model_name])
        costs.append(cost)

        judge = parse_json(groq_call(
            f'Question: {prompt}\nResponse: {answer}', model='llama-3.3-70b-versatile',
            system=JUDGE_SYS
        ).choices[0].message.content)
        avg = np.mean([judge.get('helpfulness',0), judge.get('correctness',0), judge.get('completeness',0)])
        scores.append(avg)

    q = np.mean(scores)
    c = np.mean(costs)
    results[model_name] = {'quality': q, 'cost': c}
    print(f'\n  {model_name}: Quality={q:.2f}/5  Cost/query=${c:.8f}')

# Verdict
r8, r70 = results['8B (Fast)'], results['70B (Smart)']
cost_ratio = r70['cost'] / r8['cost']
quality_ratio = r70['quality'] / r8['quality']
verdict = 'WORTH IT' if quality_ratio > cost_ratio else 'NOT WORTH IT'

print(f'\n  70B costs {cost_ratio:.1f}x more, delivers {quality_ratio:.2f}x the quality.')
print(f'  Verdict: 70B upgrade is {verdict} on a quality-per-dollar basis.')

Quality per Dollar

  8B (Fast): Quality=4.83/5  Cost/query=$0.00003443

  70B (Smart): Quality=4.75/5  Cost/query=$0.00033414

  70B costs 9.7x more, delivers 0.98x the quality.
  Verdict: 70B upgrade is NOT WORTH IT on a quality-per-dollar basis.
